In [1]:
import nltk
nltk.download('stopwords')

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import pandas as pd
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [2]:
def apply_low(text):
    return text.lower()


def apply_rsw(text):
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)


def apply_stm(text):
    words = text.split()
    words = [stemmer.stem(w) for w in words]
    return " ".join(words)


def preprocess(text, steps):
    if "LOW" in steps:
        text = apply_low(text)
    if "RSW" in steps:
        text = apply_rsw(text)
    if "STM" in steps:
        text = apply_stm(text)
    return text

In [7]:
def run_experiment(df, steps):
    df_copy = df.copy()

    df_copy["clean"] = df_copy["note"].astype(str).apply(lambda x: preprocess(x, steps))

    X_train, X_test, y_train, y_test = train_test_split(
        df_copy["clean"], df_copy["specialty"], test_size=0.2, random_state=42, stratify=df_copy["specialty"]
    )

    vectorizer = TfidfVectorizer()
    X_train_vec = vectorizer.fit_transform(X_train)
    X_test_vec = vectorizer.transform(X_test)

    svm = LinearSVC()
    svm.fit(X_train_vec, y_train)

    y_pred = svm.predict(X_test_vec)

    print(f"Steps: {steps}")
    print(classification_report(y_test, y_pred))


In [6]:
df = pd.read_csv("/content/synthetic_medical_notes.csv")

step_sets = [
    ["LOW"],
    ["RSW"],
    ["STM"],
    ["LOW", "RSW"],
    ["LOW", "STM"],
    ["RSW", "STM"],
    ["LOW", "RSW", "STM"]
]

for steps in step_sets:
    run_experiment(df, steps)

===== Steps: ['LOW'] =====
                  precision    recall  f1-score   support

      Cardiology       0.76      0.77      0.77        70
     Dermatology       0.83      0.89      0.86        71
             ENT       0.78      0.76      0.77        70
   Endocrinology       0.73      0.66      0.69        68
Gastroenterology       0.64      0.64      0.64        66
      Nephrology       0.79      0.78      0.78        72
       Neurology       0.77      0.64      0.70        74
        Oncology       0.73      0.79      0.76        77
      Pediatrics       0.65      0.66      0.65        73
      Psychiatry       0.75      0.75      0.75        72
     Pulmonology       0.68      0.64      0.66        77
    Rheumatology       0.75      0.81      0.78        70
         Surgery       0.81      0.75      0.78        68
         Urology       0.63      0.74      0.68        72

        accuracy                           0.73      1000
       macro avg       0.73      0.73      